# Step 3: Train DenseNet121
Runtime must be T4 GPU. If the session is new: re-clone the repo, `pip install monai dvc`, mount Drive, then `dvc pull` to restore data/processed.

In [ ]:
!pip -q install --upgrade monai dvc

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
GITHUB_URL = 'https://github.com/YOUR_USER/medai.git'  # <-- change this
!git clone {GITHUB_URL} /content/medai 2>/dev/null || (cd /content/medai && git pull)
%cd /content/medai

In [ ]:
# Restore the versioned data from the DVC remote (instant if data/processed already exists)
!dvc pull

In [ ]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0))

In [ ]:
from ml.train import train
model, metrics = train(epochs=6)  # about 20-30 min on T4

In [ ]:
# ROC curve plot for the report
import matplotlib.pyplot as plt, numpy as np
from sklearn.metrics import roc_curve
from ml.train import predict
from ml.dataset import CXRDataset
import torch
te = torch.utils.data.DataLoader(CXRDataset('data/splits/test.csv','data/processed'), batch_size=64)
p, t = predict(model, te, 'cuda')
f, r, _ = roc_curve(t, p)
plt.plot(f, r); plt.plot([0,1],[0,1],'--'); plt.xlabel('False positive rate'); plt.ylabel('True positive rate')
plt.title(f"Test ROC (AUROC={metrics['test_auroc']:.3f})"); plt.savefig('/content/drive/MyDrive/medai_ckpt/roc.png'); plt.show()